# JSON Data

I've assembled all the Sounds of San Anto data in Excel spreadsheet format. I would also like to share the data in JSON format. This is particularly relevant for the venue locations, which should be shared in GeoJSON format.

In [1]:
import pandas as pd
import json

In [25]:
data = pd.ExcelFile('output-data/sounds-of-san-anto.xlsx')
data.sheet_names

['artists',
 'venues',
 'locations',
 'concerts',
 'genres',
 'artists_and_genres',
 'genres_and_subgenres']

## Artists

JSON is fundamentally a computational data exchange format, so human readability is less important than machine readability. Accordingly, I'll format the column names as JSON-standard lower-case or snake-case object keys. I also will not use pretty-print indentation, making the final data object more compact.

In [26]:
artists = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='artists')
artists.info()

<class 'pandas.DataFrame'>
RangeIndex: 20320 entries, 0 to 20319
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Index             20320 non-null  int64
 1   Artist            20320 non-null  str  
 2   First Appearance  20320 non-null  str  
 3   Last Appearance   20320 non-null  str  
dtypes: int64(1), str(3)
memory usage: 635.1 KB


In [27]:
artists_export = artists.rename(columns={
    'Index': 'index',
    'Artist': 'artist',
    'First Appearance': 'firstAppearance',
    'Last Appearance': 'lastAppearance'
})

# artists_export.to_json('output-data/json/artists.json', orient="records")

## Venues

In the Excel file, intended for human consumption, the venues are sorted in order of chronological appearance in the data. I think that for computational purposes, ordering by index makes more sense. I will also rename the columns to generate more machine-readable JSON keys.

In [28]:
venues = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='venues')
venues.info()

<class 'pandas.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Index             2971 non-null   int64
 1   Venue             2971 non-null   str  
 2   First Appearance  2602 non-null   str  
 3   Last Appearance   2602 non-null   str  
dtypes: int64(1), str(3)
memory usage: 93.0 KB


In [29]:
venues_sorted = venues.sort_values("Index").reset_index(drop=True)

venues_export = venues_sorted.rename(columns={
    'Index': 'index',
    'Venue': 'venue',
    'First Appearance': 'firstAppearance',
    'Last Appearance': 'lastAppearance'
})

# venues_export.to_json('output-data/json/venues.json', orient="records")

## Venue Locations

The venue locations should be exported as GeoJSON instead of normal JSON. This will allow the file to be used directly in GIS applications. I'll load the data, then I'll generate a JSON-to-GeoJSON conversion method and use that to generate the GeoJSON file. 

In [30]:
venue_locations = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='locations')
venue_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       974 non-null    str    
 4   state      974 non-null    str    
 5   zip        974 non-null    int64  
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(2), int64(2), str(4)
memory usage: 61.0 KB


In [31]:
def venues_to_geojson(df):
    features = []
    for _, row in df.iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["longitude"], row["latitude"]]
            },
            "properties": {
                "id": row["id"],
                "name": row["name"],
                "address": row["address"],
                "city": row["city"],
                "state": row["state"],
                "zip": row["zip"]
            }
        }
        features.append(feature)

    return {
        "type": "FeatureCollection",
        "features": features
    }

In [32]:
venues_geojson = venues_to_geojson(venue_locations)

# with open("output-data/json/venueLocations.geojson", "w") as f:
#     json.dump(venues_geojson, f)

## Concerts

To export the concert list to JSON, I'll just convert the column names into more machine-friendly JSON key names and bring the index column forward so it becomes the first key in each object.

In [2]:
concerts = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='concerts')
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Month           37892 non-null  str  
 1   Day             37892 non-null  int64
 2   Year            37892 non-null  int64
 3   Venue           37892 non-null  str  
 4   Event_Artists   37892 non-null  str  
 5   Artist_Formula  37892 non-null  str  
 6   Event_Formulas  7723 non-null   str  
 7   concertIndex    37892 non-null  int64
 8   artistIndex     37892 non-null  int64
 9   venueIndex      37892 non-null  int64
dtypes: int64(5), str(5)
memory usage: 2.9 MB


In [4]:
concerts_export = concerts.rename(columns={
    'Month': 'month',
    'Day': 'day',
    'Year': 'year',
    'Venue': 'venue',
    'Event_Artists': 'eventArtists',
    'Artist_Formula': 'artistFormula',
    'Event_Formulas': 'eventFormulas'
})
concerts_export = concerts_export[['concertIndex', 'artistIndex', 'venueIndex', 'month', 'day', 'year', 'venue', 'eventArtists', 'artistFormula', 'eventFormulas']]

In [6]:
# concerts_export.to_json('output-data/json/concerts.json', orient='records')

## Genres

For genres, I will follow the same procedure. I'll just rename the columns so they form more easily machine-readable JSON keys. Then I'll export them as JSON files. For the genre hierarchy file, I'll just re-export the original, with a new name.

In [38]:
genres = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='genres')
genres_and_artists = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='artists_and_genres')
genre_hierarchy = pd.read_json('input-data/genre_hierarchy_5.json')

In [39]:
genres.info()

<class 'pandas.DataFrame'>
RangeIndex: 1045 entries, 0 to 1044
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Index   1045 non-null   int64
 1   Genre   1045 non-null   str  
dtypes: int64(1), str(1)
memory usage: 16.5 KB


In [40]:
genres_export = genres.rename(columns={
    "Index": "index",
    "Genre": "genre"
})

In [42]:
# genres_export.to_json('output-data/json/genres.json', orient='records')

In [43]:
artists_genres = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='artists_and_genres')
artists_genres.info()

<class 'pandas.DataFrame'>
RangeIndex: 10095 entries, 0 to 10094
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Artist       10095 non-null  str  
 1   Genre        10095 non-null  str  
 2   artistIndex  10095 non-null  int64
 3   genreIndex   10095 non-null  int64
dtypes: int64(2), str(2)
memory usage: 315.6 KB


In [44]:
artists_genres_export = artists_genres.rename(columns={
    'Artist': 'artist',
    'Genre': 'genre'
})

In [47]:
# artists_genres_export.to_json('output-data/json/artistsGenres.json', orient='records')

In [49]:
# genre_hierarchy.to_json('output-data/json/genresSubGenres.json', orient='records')